# Filter data: REDS-Index and REDS-Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from hk1_r12ter_analysis.config import INTERIM_DATA_DIR, logger
from hk1_r12ter_analysis.data.filtering import (
    filter_low_abundance,
    filter_low_quality_by_missingness,
    filter_low_variance,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data and filter

In [ ]:
# Low-quality filtering
missingness_percent = 0.5
use_groupwise_threshold = False

# Low-variance filtering
low_variance_method = None
low_variance_percent = 0

# Low-abundance filtering
low_abundance_method = None
low_abundance_percent = 0

# DataFrame format axis=0: (Samples, Features) or axis=1: (Features, Samples)
axis = 0

input_dirpath = INTERIM_DATA_DIR / "REDS" / "Formatted"
output_dirpath = INTERIM_DATA_DIR / "REDS" / "Filtered"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Load REDS Index data

In [ ]:
source_type = "REDS-Index"
data_types = [
    "Metabolomics",
    "Proteomics",
]

sample_key = "INDEX DONOR ID"
group_key = None

### Filter

In [ ]:
index_cols = [key for key in [sample_key, group_key] if key]
for data_type in data_types:
    # Load data
    filename_args = (source_type, "Donor", "RBC", data_type)
    logger.debug(f"Processing data for '{'-'.join(filename_args)}'...")
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)

    # Apply low-quality filter
    if missingness_percent not in {None, 0, 1}:
        df = filter_low_quality_by_missingness(
            df,
            percent=missingness_percent,
            use_groupwise_threshold=use_groupwise_threshold,
            group_key=group_key,
            axis=axis,
        )
    # TODO Apply low-repeatability filter
    # Apply low variance filter
    if low_variance_method is not None:
        df = filter_low_variance(
            df, method=low_variance_method, percent=low_variance_percent, axis=axis
        )
    # Apply low-abundance filter
    if low_abundance_method is not None:
        df = filter_low_abundance(
            df, method=low_abundance_method, percent=low_abundance_percent, axis=axis
        )

    save_dataframe_to_file(df, output_dirpath / filename, index=True)
    logger.info(f"Processed data for '{'-'.join(filename_args)}'.")

### Load REDS Recall data

In [ ]:
source_type = "REDS-Recall"
data_types = [
    "Proteomics",
    "Metabolomics",
    "Lipidomics",
    # "Lipidomics-DegUnsat",
    # "Lipidomics-LipidClass",
    "TraceElements",
    # "Vesicles-mtDNA",
]
sample_key = "RECALL DONOR ID"
group_key = "Day"

### Filter

In [ ]:
index_cols = [key for key in [sample_key, group_key] if key]
for data_type in data_types:
    # Load data
    filename_args = (source_type, "Donor", "RBC", data_type)
    logger.debug(f"Processing data for '{'-'.join(filename_args)}'...")
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)

    # Apply low-quality filter
    if missingness_percent not in {None, 0, 1}:
        df = filter_low_quality_by_missingness(
            df,
            percent=missingness_percent,
            use_groupwise_threshold=use_groupwise_threshold,
            group_key=group_key,
            axis=axis,
        )
    # TODO Apply low-repeatability filter
    # Apply low variance filter
    if low_variance_method is not None:
        df = filter_low_variance(
            df, method=low_variance_method, percent=low_variance_percent, axis=axis
        )
    # Apply low-abundance filter
    if low_abundance_method is not None:
        df = filter_low_abundance(
            df, method=low_abundance_method, percent=low_abundance_percent, axis=axis
        )

    save_dataframe_to_file(df, output_dirpath / filename, index=True)
    logger.info(f"Processed data for '{'-'.join(filename_args)}'.")